# GO:BP ORA Analysis: pathway coverage across sample sizes

**Environment:** `clamp-analyses`

For each CLAMP model (CLAMPfull and CLAMPbase) across all coverage levels (1%, 5%, 10%, 25%, 50%, 75%, 100%) and 3 seeds, this notebook:

1. Loads the Z matrix (gene loadings per LV) for all 3 seeds at a given coverage level.
2. For each LV, selects the top 1% genes by descending loading.
3. Runs `enrichGO(ont = "BP")` per LV using all model genes as universe.
4. Combines full unfiltered ORA results across all seeds and LVs.
5. Saves one RDS per coverage level per model type.

FDR filtering (0.05 / 0.01) and coverage computation are done in `01_bp_coverage_plot.ipynb`.

In [ ]:
library(here)
library(clusterProfiler)
library(org.Hs.eg.db)
library(parallel)

## Paths

In [ ]:
models_dir <- here("output/01_model_building/04_archs4/06_bp_coverage_rshall")
output_dir <- here("output/03_model_biology/00_archs4/00_pathway_coverage_bp/00_bp_ora_analysis")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(output_dir, "CLAMPbase"),  recursive = TRUE, showWarnings = FALSE)

## Coverage level specs

In [ ]:
coverage_specs <- list(
  list(pct = 1,   dir = "00_bp_coverage_hall_rs_01"),
  list(pct = 5,   dir = "01_bp_coverage_hall_rs_05"),
  list(pct = 10,  dir = "02_bp_coverage_hall_rs_10"),
  list(pct = 25,  dir = "03_bp_coverage_hall_rs_25"),
  list(pct = 50,  dir = "04_bp_coverage_hall_rs_50"),
  list(pct = 75,  dir = "05_bp_coverage_hall_rs_75"),
  list(pct = 100, dir = "06_bp_coverage_hall_rs_100")
)

seeds <- 1:3

## Helper: run enrichGO for one model (one seed, one coverage level)

In [ ]:
run_ora_one_seed <- function(z_path, seed_id, n_cores = 1L) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  n_top          <- ceiling(0.01 * nrow(Z))
  n_lvs          <- ncol(Z)
  lv_names       <- colnames(Z)

  top_genes_per_lv <- apply(Z, 2, function(lv) {
    universe_genes[order(lv, decreasing = TRUE)[seq_len(n_top)]]
  })
  # apply drops dim when ncol(Z) == 1; restore so [, i] indexing works
  if (is.null(dim(top_genes_per_lv))) {
    dim(top_genes_per_lv) <- c(length(top_genes_per_lv), 1L)
  }
  rm(Z)
  gc()  # free Z and apply's matrix copy before forking

  ora_list <- parallel::mclapply(seq_len(n_lvs), function(i) {
    res <- tryCatch(
      clusterProfiler::enrichGO(
        gene          = top_genes_per_lv[, i],
        universe      = universe_genes,
        OrgDb         = org.Hs.eg.db,
        keyType       = "SYMBOL",
        ont           = "BP",
        pAdjustMethod = "BH",
        pvalueCutoff  = 1,
        qvalueCutoff  = 1,
        minGSSize     = 10,
        maxGSSize     = 500
      ),
      error = function(e) NULL
    )
    if (is.null(res) || nrow(as.data.frame(res)) == 0) return(NULL)
    df      <- as.data.frame(res)[, c("ID", "p.adjust")]
    df$LV   <- lv_names[i]
    df$seed <- seed_id
    df
  }, mc.cores = n_cores)

  ora_list <- Filter(function(x) !is.null(x) && !inherits(x, "try-error"), ora_list)

  ora_df <- do.call(rbind, ora_list)
  if (is.null(ora_df) || nrow(ora_df) == 0) return(NULL)

  list(ora_df = ora_df, seed = seed_id, n_lvs = n_lvs)
}

## Run ORA: CLAMPfull

In [ ]:
for (spec in coverage_specs) {
  cache_path <- file.path(output_dir, "CLAMPfull", sprintf("pct%d.rds", spec$pct))

  if (file.exists(cache_path)) {
    message(sprintf("Skipping CLAMPfull pct%d (cached)", spec$pct))
    next
  }

  message(sprintf("Running ORA: CLAMPfull pct%d", spec$pct))

  seed_results <- lapply(seeds, function(s) {
    seed_dir <- file.path(models_dir, spec$dir,
                          sprintf("hall_coverage_rs%d_seed_%d", spec$pct, s))
    z_path   <- file.path(seed_dir, "CLAMPfull_hall", "Z.csv")
    b_path   <- file.path(seed_dir, "CLAMPfull_hall", "B.csv")

    if (!file.exists(z_path)) {
      warning("Z.csv not found: ", z_path); return(NULL)
    }

    # read n_samples from B.csv header only (no full model load)
    n_samples <- if (file.exists(b_path)) {
      ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
    } else NA_integer_

    res <- run_ora_one_seed(z_path, seed_id = s, n_cores = 4L)
    if (is.null(res)) return(NULL)
    res$n_samples <- n_samples
    res
  })
  seed_results <- Filter(Negate(is.null), seed_results)

  if (length(seed_results) == 0) {
    warning("No results for CLAMPfull pct", spec$pct); next
  }

  ora_df <- do.call(rbind, lapply(seed_results, `[[`, "ora_df"))
  rownames(ora_df) <- NULL

  meta <- do.call(rbind, lapply(seed_results, function(r) {
    data.frame(seed = r$seed, n_samples = r$n_samples, n_lvs = r$n_lvs)
  }))

  saveRDS(
    list(ora_df = ora_df, meta = meta, n_total_bp = length(unique(ora_df$ID))),
    cache_path
  )
  message(sprintf("  Saved: %s (%d rows, %d unique terms)",
                  basename(cache_path), nrow(ora_df), length(unique(ora_df$ID))))
}

message("CLAMPfull done.")

## Run ORA: CLAMPbase

In [ ]:
for (spec in coverage_specs) {
  cache_path <- file.path(output_dir, "CLAMPbase", sprintf("pct%d.rds", spec$pct))

  if (file.exists(cache_path)) {
    message(sprintf("Skipping CLAMPbase pct%d (cached)", spec$pct))
    next
  }

  message(sprintf("Running ORA: CLAMPbase pct%d", spec$pct))

  seed_results <- lapply(seeds, function(s) {
    seed_dir <- file.path(models_dir, spec$dir,
                          sprintf("hall_coverage_rs%d_seed_%d", spec$pct, s))
    z_path   <- file.path(seed_dir, "CLAMPbase", "Z.csv")
    b_path   <- file.path(seed_dir, "CLAMPbase", "B.csv")

    if (!file.exists(z_path)) {
      warning("Z.csv not found: ", z_path); return(NULL)
    }

    n_samples <- if (file.exists(b_path)) {
      ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
    } else NA_integer_

    res <- run_ora_one_seed(z_path, seed_id = s, n_cores = 4L)
    if (is.null(res)) return(NULL)
    res$n_samples <- n_samples
    res
  })
  seed_results <- Filter(Negate(is.null), seed_results)

  if (length(seed_results) == 0) {
    warning("No results for CLAMPbase pct", spec$pct); next
  }

  ora_df <- do.call(rbind, lapply(seed_results, `[[`, "ora_df"))
  rownames(ora_df) <- NULL

  meta <- do.call(rbind, lapply(seed_results, function(r) {
    data.frame(seed = r$seed, n_samples = r$n_samples, n_lvs = r$n_lvs)
  }))

  saveRDS(
    list(ora_df = ora_df, meta = meta, n_total_bp = length(unique(ora_df$ID))),
    cache_path
  )
  message(sprintf("  Saved: %s (%d rows, %d unique terms)",
                  basename(cache_path), nrow(ora_df), length(unique(ora_df$ID))))
}

message("CLAMPbase done.")